Consolidation pipeline

In [1]:
import pandas as pd

In [2]:
# Import all CSV files
us = pd.read_csv("us_region_sales-1.csv")
eu = pd.read_csv("eu_region_sales.csv")
apac = pd.read_csv("apac_region_sales-1.csv")
rates = pd.read_csv("exchange_rates.csv")
products = pd.read_csv("product_master.csv")
sales = pd.read_csv("Consolidated Sales.csv")

In [3]:
# Standardize transaction dates
us["Date"] = pd.to_datetime(us["Sale_Date"], format="%m/%d/%Y", errors="coerce")
eu["Date"] = pd.to_datetime(eu["date"], format="%Y-%m-%d", errors="coerce")
apac["Date"] = pd.to_datetime(apac["txn_date_ddmmyyyy"], format="%d-%m-%Y", errors="coerce")
rates["Date"] = pd.to_datetime(rates["rate_date"], format="%Y-%m-%d", errors="coerce")

In [4]:
# Check for invalid dates
print("US:", us["Date"].isna().sum())
print("EU:", eu["Date"].isna().sum())
print("APAC:", apac["Date"].isna().sum())
print("Exchange Rates:", rates["Date"].isna().sum())

US: 4
EU: 3
APAC: 3
Exchange Rates: 0


In [5]:
# Display invalid US dates
us[us["Date"].isna()][["Order_No", "Sale_Date"]]

,Order_No,Sale_Date
130,US-100203,17/02/2026
336,US-100527,13/05/2026
341,US-100534,14/05/2026
618,US-100967,20/08/2026


In [6]:
# Display invalid EU dates
eu[eu["Date"].isna()][["order_ref", "date"]]

,order_ref,date
55,EU-200090,30.01.2026
208,EU-200336,12.04.2026
464,EU-200743,31.07.2026


In [7]:
# Display invalid APAC dates
apac[apac["Date"].isna()][["txn_ref", "txn_date_ddmmyyyy"]]

,txn_ref,txn_date_ddmmyyyy
104,AP-300159,2026-02-26
209,AP-300341,2026-04-29
434,AP-300670,2026-08-23


In [8]:
# Calculate original local revenue
us["Local_Revenue"] = us["Qty_Sold"] * us["Unit_Price_USD"]
eu["Local_Revenue"] = eu["line_total_eur_incl_vat"]
apac["Local_Revenue"] = apac["qty"] * apac["unit_price_local"]

In [9]:
# Identify duplicate exchange rates
rates[rates.duplicated(subset=["Date", "currency"], keep=False)]

,rate_date,currency,usd_per_unit,Date
195,2026-02-19,EUR,1.077520,2026-02-19
196,2026-02-19,EUR,1.206820,2026-02-19
859,2026-08-05,JPY,0.007459,2026-08-05
860,2026-08-05,JPY,0.006660,2026-08-05


In [10]:
# Keep the first exchange rate for each date and currency
original_rates = rates.drop_duplicates(subset=["Date", "currency"], keep="first")

In [11]:
# Assign EUR currency to European sales
eu["currency"] = "EUR"

# Match European exchange rates
eu_usd = eu.merge(
    original_rates[["Date", "currency", "usd_per_unit"]],
    on=["Date", "currency"],
    how="left"
)

# Match APAC exchange rates
apac_usd = apac.merge(
    original_rates[["Date", "currency", "usd_per_unit"]],
    on=["Date", "currency"],
    how="left"
)

In [12]:
# Convert local revenue to USD
us["In_USD"] = us["Local_Revenue"]
eu_usd["In_USD"] = eu_usd["Local_Revenue"] * eu_usd["usd_per_unit"]
apac_usd["In_USD"] = apac_usd["Local_Revenue"] * apac_usd["usd_per_unit"]

In [13]:
# Count missing exchange rates for valid dates
print("EU:", eu_usd["usd_per_unit"].isna().mul(eu_usd["Date"].notna()).sum())
print("APAC:", apac_usd["usd_per_unit"].isna().mul(apac_usd["Date"].notna()).sum())

EU: 10
APAC: 9


In [14]:
# Calculate original USD revenue by region
us_total = us.loc[us["Date"].notna(), "In_USD"].sum()
eu_total = eu_usd.loc[eu_usd["Date"].notna(), "In_USD"].sum()
apac_total = apac_usd.loc[apac_usd["Date"].notna(), "In_USD"].sum()

# Calculate global revenue
global_total = us_total + eu_total + apac_total

# Display results
print("US Revenue:", round(us_total, 2))
print("EU Revenue:", round(eu_total, 2))
print("APAC Revenue:", round(apac_total, 2))
print("Total In USD Revenue:", round(global_total, 2))

US Revenue: 80483.55
EU Revenue: 89240.27
APAC Revenue: 111188.19
Total In USD Revenue: 280912.01


In [15]:
# Review surrounding EUR exchange rates
rates[(rates["currency"] == "EUR") &
      (rates["Date"].between("2026-02-16", "2026-02-22"))]

,rate_date,currency,usd_per_unit,Date
183,2026-02-16,EUR,1.07607,2026-02-16
187,2026-02-17,EUR,1.07759,2026-02-17
191,2026-02-18,EUR,1.07595,2026-02-18
195,2026-02-19,EUR,1.07752,2026-02-19
196,2026-02-19,EUR,1.20682,2026-02-19
200,2026-02-20,EUR,1.07959,2026-02-20
204,2026-02-21,EUR,1.08062,2026-02-21
208,2026-02-22,EUR,1.08264,2026-02-22


In [16]:
# Review surrounding JPY exchange rates
rates[(rates["currency"] == "JPY") &
      (rates["Date"].between("2026-08-02", "2026-08-08"))]

,rate_date,currency,usd_per_unit,Date
848,2026-08-02,JPY,0.006673,2026-08-02
852,2026-08-03,JPY,0.006686,2026-08-03
855,2026-08-04,JPY,0.006680,2026-08-04
859,2026-08-05,JPY,0.007459,2026-08-05
860,2026-08-05,JPY,0.006660,2026-08-05
864,2026-08-06,JPY,0.006673,2026-08-06
868,2026-08-07,JPY,0.006700,2026-08-07
872,2026-08-08,JPY,0.006690,2026-08-08


In [17]:
# Remove incorrect duplicate exchange rates
clean_rates = rates.copy()

clean_rates = clean_rates.drop(859)

clean_rates = clean_rates[
    ~((clean_rates["currency"] == "EUR") &
      (clean_rates["Date"] == "2026-02-19") &
      (clean_rates["usd_per_unit"] == 1.206820))
]

In [18]:
# Check remaining duplicate exchange rates
clean_rates.duplicated(subset=["Date", "currency"]).sum()

np.int64(0)

In [19]:
# Fill missing daily exchange rates
clean_rates = clean_rates.pivot(
    index="Date",
    columns="currency",
    values="usd_per_unit"
)

clean_rates = clean_rates.reindex(
    pd.date_range("2026-01-01", "2026-09-30")
)

clean_rates = clean_rates.interpolate()

clean_rates = clean_rates.stack().reset_index()

clean_rates.columns = ["Date", "currency", "usd_per_unit"]

In [20]:
# Correct abnormal exchange rates
clean_rates.loc[
    (clean_rates["Date"] == "2026-01-17") &
    (clean_rates["currency"] == "EUR"),
    "usd_per_unit"
] = 1.077415

clean_rates.loc[
    (clean_rates["Date"] == "2026-08-07") &
    (clean_rates["currency"] == "EUR"),
    "usd_per_unit"
] = 1.080390

clean_rates.loc[
    (clean_rates["Date"] == "2026-05-04") &
    (clean_rates["currency"] == "SGD"),
    "usd_per_unit"
] = 0.739360

In [21]:
# Combine year, month and day into one date
sales["Date"] = pd.to_datetime(
    sales["Year"].astype(str) + "-" +
    sales["Month"] + "-" +
    sales["Day"].astype(str),
    format="%Y-%B-%d",
    errors="coerce"
)

In [22]:
# Correct incorrectly interpreted EU date
sales.loc[
    sales["Order_ID"] == "EU-200336", "Date"
] = pd.Timestamp("2026-04-12")

# Verify corrected date
sales.loc[sales["Order_ID"] == "EU-200336",
          ["Order_ID", "Raw_date", "Date"]]

,Order_ID,Raw_date,Date
1061,EU-200336,12.04.2026,2026-04-12


In [23]:
# Identify duplicate order IDs
sales[sales.duplicated(subset=["Order_ID"], keep=False)][
    ["Order_ID", "Region", "Date", "City",
     "Product_Name", "Qty", "Raw_local_amount"]
].sort_values("Order_ID")

,Order_ID,Region,Date,City,Product_Name,Qty,Raw_local_amount
223,US-100183,EU,2026-02-14,Berlin,Corner Bracket 3in,17,52.87
432,US-100183,US,2026-02-14,Chicago,Corner Bracket 3in,17,47.60
58,US-100311,US,2026-03-17,Atlanta,Locking Cam Latch,4,14.80
1395,US-100311,EU,2026-03-17,Rotterdam,Locking Cam Latch,4,16.44
285,US-100530,EU,2026-05-14,Berlin,Soft-Close Hinge,60,861.00
811,US-100530,US,2026-05-14,Denver,Soft-Close Hinge,60,775.20
292,US-100562,EU,2026-05-19,Berlin,Soft-Close Hinge,3,43.05
630,US-100562,US,2026-05-19,Dallas,Soft-Close Hinge,3,38.76
884,US-100883,US,2026-08-02,Denver,Cabinet Pull - Matte Black,17,83.30
1269,US-100883,EU,2026-08-02,Milan,Cabinet Pull - Matte Black,17,92.48


In [24]:
# Identify duplicate order IDs
duplicate_ids = sales.loc[
    sales["Order_ID"].duplicated(keep=False), "Order_ID"
]

# Remove suspected duplicate EU transactions
sales = sales[~(
    (sales["Region"] == "EU") &
    (sales["Order_ID"].isin(duplicate_ids))
)].copy()

# Check remaining duplicate order IDs
sales["Order_ID"].duplicated().sum()

np.int64(0)

In [25]:
# Correct inconsistent tax percentages
sales.loc[sales["Tax_pct"].between(0.01, 0.99), "Tax_pct"] *= 100

# Assign regional tax percentages
sales.loc[sales["Region"] == "US", "Tax_pct"] = 0
sales.loc[sales["Region"] == "EU", "Tax_pct"] = 20

In [26]:
# Review APAC tax rates by currency
sales[sales["Region"] == "APAC"].groupby(
    ["Currency", "Tax_pct"]
).size()

Currency  Tax_pct
AUD       0.0          2
          10.0       208
JPY       10.0       127
SGD       0.0          1
          9.0        163
          10.0         5
dtype: int64

In [27]:
# Calculate revenue excluding taxes
sales["Net_Local_Revenue"] = sales["Raw_local_amount"] / (
    1 + sales["Tax_pct"] / 100
)

In [28]:
# Match corrected exchange rates
final_sales = sales.merge(
    clean_rates,
    left_on=["Date", "Currency"],
    right_on=["Date", "currency"],
    how="left"
)

# USD exchange rate is 1
final_sales.loc[final_sales["Currency"] == "USD", "usd_per_unit"] = 1

# Calculate net revenue in USD
final_sales["Net_Revenue_USD"] = (
    final_sales["Net_Local_Revenue"] * final_sales["usd_per_unit"]
)

In [29]:
# Check for missing values
print(final_sales[["Date", "Tax_pct", "usd_per_unit",
                   "Net_Revenue_USD"]].isna().sum())

# Check dates outside the reporting period
print("Out-of-period dates:",
      (~final_sales["Date"].between("2026-01-01", "2026-09-30")).sum())

Date               0
Tax_pct            0
usd_per_unit       4
Net_Revenue_USD    4
dtype: int64
Out-of-period dates: 6


In [30]:
# Display net revenue by region
final_sales.groupby("Region")["Net_Revenue_USD"].sum().round(2)

Region
APAC    102699.74
EU       74082.51
US       80694.66
Name: Net_Revenue_USD, dtype: float64

In [31]:
# Save excluded transactions for documentation
excluded_sales = final_sales[
    ~final_sales["Date"].between("2026-01-01", "2026-09-30")
].copy()

# Keep transactions within the reporting period
final_sales = final_sales[
    final_sales["Date"].between("2026-01-01", "2026-09-30")
].copy()

# Check remaining missing exchange rates
print(final_sales["usd_per_unit"].isna().sum())

0


In [32]:
# Calculate corrected gross USD revenue
final_sales["Gross_USD"] = (final_sales["Raw_local_amount"] * final_sales["usd_per_unit"])
corrected_gross = final_sales["Gross_USD"].sum()

In [33]:
# Calculate tax adjustments by region
final_sales["Tax_Adjustment"] = (final_sales["Net_Revenue_USD"] - final_sales["Gross_USD"])
eu_tax = final_sales.loc[final_sales["Region"] == "EU", "Tax_Adjustment"].sum()
apac_tax = final_sales.loc[final_sales["Region"] == "APAC", "Tax_Adjustment"].sum()

In [34]:
# Compare original and corrected exchange rates
comparison = final_sales.merge(
    original_rates[["Date", "currency", "usd_per_unit"]],
    left_on=["Date", "Currency"], right_on=["Date", "currency"],
    how="left", suffixes=("", "_original"))

comparison.loc[comparison["Currency"] == "USD", "usd_per_unit_original"] = 1

# Calculate exchange rate adjustments
rate_adjustment = (comparison["Raw_local_amount"] * (comparison["usd_per_unit"] - comparison["usd_per_unit_original"])).sum()

missing_adjustment = comparison.loc[comparison["usd_per_unit_original"].isna(), "Gross_USD"].sum()

print("Exchange Rate Corrections:", round(rate_adjustment, 2))
print("Missing Exchange Rates:", round(missing_adjustment, 2))

Exchange Rate Corrections: 82.99
Missing Exchange Rates: 2203.34


In [35]:
# Calculate remaining data adjustments
data_adjustment = corrected_gross - global_total - rate_adjustment - missing_adjustment

print("Data Adjustments:", round(data_adjustment, 2))

Data Adjustments: -988.34


In [36]:
# Create revenue bridge
bridge = pd.DataFrame({
    "Description": ["Original Raw Revenue in USD", "Data Adjustments",
                    "Exchange Rate Corrections", "Missing Exchange Rates",
                    "EU VAT Adjustment", "APAC Sales Tax Adjustment",
                    "Final Net USD Revenue"],
    "Amount_USD": [global_total, data_adjustment, rate_adjustment,
                   missing_adjustment, eu_tax, apac_tax,
                   final_sales["Net_Revenue_USD"].sum()]
})

bridge.round(2)

,Description,Amount_USD
0,Original Raw Revenue in USD,280912.01
1,Data Adjustments,-988.34
2,Exchange Rate Corrections,82.99
3,Missing Exchange Rates,2203.34
4,EU VAT Adjustment,-14816.50
5,APAC Sales Tax Adjustment,-10045.78
6,Final Net USD Revenue,257347.71


In [37]:
# Export final results
final_sales.to_csv("Final_Consolidated_Sales.csv", index=False)
bridge.round(2).to_csv("Revenue_Bridge.csv", index=False)
excluded_sales.to_csv("Excluded_Transactions.csv", index=False)